# Q.4 ML + Quantum + Parallel Computing
Crop assignment: 4 plots, 3 crops. Run all cells top to bottom (Runtime → Run all).

## Setup, imports and helpers

In [1]:
import itertools
import time
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from scipy.optimize import minimize
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR

SEED = 23
CROPS = ["Wheat", "Rice", "Maize"]
N_PLOTS, N_CROPS = 4, 3
NQ = N_PLOTS * N_CROPS                      # 12 yes/no variables
NEIGHBOURS = [(0, 1), (1, 2), (2, 3)]       # plots in a row: 0-1-2-3


def var(p, c):
    """index of binary variable x[p,c] ('plot p gets crop c')"""
    return p * N_CROPS + c

## (a) Machine Learning: Support Vector Regression

In [2]:
def part_a():
    print("=" * 60, "\n(a) MACHINE LEARNING - Support Vector Regression\n", "=" * 60)
    rng = np.random.default_rng(SEED)
    n = 600
    nitrogen = rng.uniform(20, 120, n)       # kg/ha
    rainfall = rng.uniform(300, 1200, n)     # mm
    area = rng.uniform(1, 10, n)             # hectares
    crop = rng.integers(0, 3, n)
    prev = rng.integers(0, 3, n)

    base = np.array([3.0, 4.0, 3.5])[crop]
    n_eff = np.array([0.020, 0.012, 0.025])[crop] * nitrogen
    r_eff = np.array([0.0030, 0.0045, 0.0020])[crop] * rainfall
    # rotation bonus: growing a different crop than last season helps
    rot = np.where(crop == prev, -0.8, 0.4)
    y = base + n_eff + r_eff - 0.0000025 * (rainfall - 750) ** 2 \
        + 0.05 * area + rot + rng.normal(0, 0.25, n)

    X = np.column_stack([nitrogen, rainfall, area, crop, prev])
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, random_state=SEED)

    pre = ColumnTransformer([
        ("num", StandardScaler(), [0, 1, 2]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), [3, 4]),
    ])
    model = Pipeline([("pre", pre), ("svr", SVR(kernel="rbf", C=10, epsilon=0.05))])
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    rmse = np.sqrt(mean_squared_error(yte, pred))
    print(f"Train rows: {len(Xtr)}  Test rows: {len(Xte)}")
    print(f"RMSE = {rmse:.4f}   R^2 = {r2_score(yte, pred):.4f}")

    # the 4 real plots: (nitrogen, rainfall, area, previous crop)
    plots = np.array([[80, 700, 4.0, 0],
                      [50, 900, 6.0, 1],
                      [100, 500, 3.0, 2],
                      [60, 1000, 5.0, 0]], dtype=float)
    Y = np.zeros((N_PLOTS, N_CROPS))
    for p in range(N_PLOTS):
        for c in range(N_CROPS):
            row = np.array([[plots[p, 0], plots[p, 1], plots[p, 2], c, plots[p, 3]]])
            Y[p, c] = model.predict(row)[0]
    print("\nPredicted yield (t/ha) for each plot x crop:")
    print("Plot | " + " | ".join(f"{c:>6}" for c in CROPS))
    for p in range(N_PLOTS):
        print(f"  {p+1}  | " + " | ".join(f"{Y[p,c]:6.3f}" for c in range(N_CROPS)))
    return Y

## (b) QUBO formulation + brute force

In [3]:
def build_qubo(Y):
    """
    E(x) = - sum_{p,c} Y[p,c] x_pc                      (maximise yield)
           + A * sum_p (sum_c x_pc - 1)^2               (exactly one crop per plot)
           + B * sum_{(p,q) neigh} sum_c x_pc x_qc      (same crop on neighbours)
    Using x^2 = x:  A(sum_c x_pc -1)^2 = A( -sum_c x_pc + 2 sum_{c<d} x_pc x_pd + 1 )
    Constant offset = A * N_PLOTS.  E = x^T Q x + offset  (Q upper-triangular)
    """
    ymax = Y.max()
    A = B = 2.0 * ymax
    Q = np.zeros((NQ, NQ))
    for p in range(N_PLOTS):
        for c in range(N_CROPS):
            i = var(p, c)
            Q[i, i] += -Y[p, c] - A                       # yield + linear part of penalty
            for d in range(c + 1, N_CROPS):
                Q[i, var(p, d)] += 2 * A                  # one-hot coupling
    for (p, q) in NEIGHBOURS:
        for c in range(N_CROPS):
            Q[var(p, c), var(q, c)] += B                  # rotation rule
    offset = A * N_PLOTS
    return Q, offset, A, B


def all_energies(Q, offset):
    """energy of all 2^NQ bit-strings (vectorised, used by brute force & QAOA)"""
    N = 2 ** NQ
    idx = np.arange(N)
    bits = ((idx[:, None] >> (NQ - 1 - np.arange(NQ))) & 1).astype(float)  # qubit0 = MSB
    return np.einsum("ni,ij,nj->n", bits, Q, bits) + offset


def decode(k):
    s = format(k, f"0{NQ}b")
    return s, [int(s[p * N_CROPS:(p + 1) * N_CROPS].find("1")) if s[p*N_CROPS:(p+1)*N_CROPS].count("1") == 1 else None
               for p in range(N_PLOTS)]


def describe(k):
    s, a = decode(k)
    return s + "  ->  " + ", ".join(f"Plot{p+1}:{CROPS[c] if c is not None else '??'}" for p, c in enumerate(a))


def part_b(Y):
    print("\n" + "=" * 60, "\n(b) QUBO FORMULATION + BRUTE FORCE CHECK\n", "=" * 60)
    Q, offset, A, B = build_qubo(Y)
    print("QUBO:  E(x) = - sum Y[p,c] x_pc + A sum_p (sum_c x_pc - 1)^2"
          "\n              + B sum_(p,q neighbours) sum_c x_pc x_qc")
    print(f"Penalties A = B = {A:.3f}  (= 2 x max predicted yield),  constant offset = {offset:.3f}")
    np.set_printoptions(precision=2, linewidth=200, suppress=True)
    print("Q matrix (12 x 12, upper triangular):\n", Q)

    E = all_energies(Q, offset)
    kbest = int(np.argmin(E))
    print("\nBrute force over 4096 states:")
    print("  lowest energy =", round(E[kbest], 4))
    print("  best answer   =", describe(kbest))

    # independent check: enumerate only the valid 3^4 assignments directly
    best, best_val = None, -1e9
    for a in itertools.product(range(3), repeat=4):
        if any(a[p] == a[q] for p, q in NEIGHBOURS):
            continue
        v = sum(Y[p, a[p]] for p in range(4))
        if v > best_val:
            best, best_val = a, v
    print("  Direct check (valid plans only): " +
          ", ".join(f"Plot{p+1}:{CROPS[c]}" for p, c in enumerate(best)) +
          f"   total yield = {best_val:.3f}  (energy should be {-best_val:.3f})")
    _, a_qubo = decode(kbest)
    print("  QUBO optimum matches direct optimum:", tuple(a_qubo) == best)
    return Q, offset, E, kbest

## (c) Quantum: QAOA (p=1) simulator

In [4]:
def qaoa_state(E_norm, gamma, beta):
    """p=1 QAOA statevector: |+>^n -> exp(-i*gamma*E) -> RX(2*beta) on each qubit"""
    psi = np.full(2 ** NQ, 1 / np.sqrt(2 ** NQ), dtype=complex)
    psi *= np.exp(-1j * gamma * E_norm)
    c, s = np.cos(beta), -1j * np.sin(beta)
    psi = psi.reshape([2] * NQ)
    for q in range(NQ):
        a0 = np.take(psi, 0, axis=q)
        a1 = np.take(psi, 1, axis=q)
        psi = np.stack([c * a0 + s * a1, s * a0 + c * a1], axis=q)
    return psi.reshape(-1)


def part_c(E, kbest, shots=8192):
    print("\n" + "=" * 60, "\n(c) QUANTUM COMPUTING - QAOA (p = 1, NumPy simulator)\n", "=" * 60)
    En = (E - E.min()) / (E.max() - E.min())          # scale to [0,1] for stable angles

    def expval(params):
        psi = qaoa_state(En, params[0], params[1])
        return float(np.real(np.sum(np.abs(psi) ** 2 * En)))

    # coarse grid search, then local refinement
    best = (1e9, None)
    for g in np.linspace(0, 2 * np.pi, 40):
        for b in np.linspace(0, np.pi, 20):
            v = expval([g, b])
            if v < best[0]:
                best = (v, [g, b])
    res = minimize(expval, best[1], method="Nelder-Mead", options={"xatol": 1e-3, "fatol": 1e-6})
    gamma, beta = res.x
    print(f"Optimised angles: gamma = {gamma:.4f}, beta = {beta:.4f}   <E_norm> = {res.fun:.4f}")

    psi = qaoa_state(En, gamma, beta)
    probs = np.abs(psi) ** 2
    probs /= probs.sum()
    rng = np.random.default_rng(SEED)
    samples = rng.choice(len(probs), size=shots, p=probs)
    counts = np.bincount(samples, minlength=len(probs))

    k_q = int(samples[np.argmin(E[samples])])         # lowest-energy bit-string sampled
    print(f"\nShots = {shots}")
    print("QAOA best answer :", describe(k_q))
    print(f"QAOA best energy : {E[k_q]:.4f}")
    print(f"Appears {counts[k_q]} / {shots} times  ({100*counts[k_q]/shots:.2f}%)")
    print(f"Random-guess chance for one state = {100/2**NQ:.3f}%  -> QAOA boost x{probs[k_q]*2**NQ:.1f}")
    print("\nTop 5 most frequent outcomes:")
    for k in np.argsort(counts)[::-1][:5]:
        print(f"  {format(k, f'0{NQ}b')}  E = {E[k]:8.3f}  count = {counts[k]}")
    print("\nBrute force best :", describe(kbest), f"  E = {E[kbest]:.4f}")
    print("QAOA found the true optimum:", k_q == kbest)
    return k_q

## (d) Parallel computing

In [5]:
def energy_chunk(args):
    """pure-Python energy of every bit-string in [start, stop) -- the CPU-heavy part"""
    start, stop, terms, offset, repeats = args
    best_e, best_k = 1e18, -1
    for _ in range(repeats):                 # "repeated runs" to make the work measurable
        best_e, best_k = 1e18, -1
        for k in range(start, stop):
            e = offset
            for i, j, w in terms:
                if (k >> (NQ - 1 - i)) & 1 and (k >> (NQ - 1 - j)) & 1:
                    e += w
            if e < best_e:
                best_e, best_k = e, k
    return best_e, best_k


def run_parallel(terms, offset, workers, repeats):
    N = 2 ** NQ
    step = N // workers
    chunks = [(w * step, (w + 1) * step, terms, offset, repeats) for w in range(workers)]
    t0 = time.perf_counter()
    if workers == 1:
        out = [energy_chunk(chunks[0])]
    else:
        with ProcessPoolExecutor(max_workers=workers) as ex:
            out = list(ex.map(energy_chunk, chunks))
    t = time.perf_counter() - t0
    return t, min(out)


def part_d(Q, offset, repeats=15):
    print("\n" + "=" * 60, "\n(d) PARALLEL COMPUTING - ProcessPoolExecutor\n", "=" * 60)
    terms = [(i, j, Q[i, j]) for i in range(NQ) for j in range(i, NQ) if Q[i, j] != 0]
    print(f"Evaluating all 2^{NQ} = {2**NQ} bit-strings, {repeats} repeats, "
          f"equal chunks (cores available: {__import__('os').cpu_count()})")
    run_parallel(terms, offset, 2, 1)         # warm-up
    times, best = {}, None
    for w in (1, 2, 4):
        t, best = run_parallel(terms, offset, w, repeats)
        times[w] = t
    print(f"\n{'Workers':>8} {'Time (s)':>10} {'Speed-up':>10} {'Efficiency':>11}")
    for w in (1, 2, 4):
        sp = times[1] / times[w]
        print(f"{w:>8} {times[w]:>10.3f} {sp:>10.2f} {100*sp/w:>10.1f}%")
    print(f"\nBest energy found in parallel = {best[0]:.4f}  (bit-string {best[1]:0{NQ}b})")
    print("Speed-up = T1 / Tn ;  Efficiency = Speed-up / n workers")
    return times

## (e) Conclusion

In [6]:
def part_e(k_q, E, kbest):
    print("\n" + "=" * 60, "\n(e) CONCLUSION\n", "=" * 60)
    _, a = decode(kbest)
    plan = ", ".join(f"Plot{p+1}={CROPS[c]}" for p, c in enumerate(a))
    print(f"1. Final choice: {plan} (lowest QUBO energy {E[kbest]:.3f}, no neighbouring plots share a crop).")
    print(f"2. SVR turned soil/rain/area/previous-crop data into yields, and QAOA (p=1) "
          f"{'found the same optimum as brute force' if k_q == kbest else 'got close to brute force but not always exactly the optimum'}.")
    print("3. Parallel processes cut run-time on 4096 states only when the work per chunk is large; "
          "otherwise process start-up cost lowers efficiency.")

## Run everything

In [7]:
Y = part_a()
Q, offset, E, kbest = part_b(Y)
k_q = part_c(E, kbest)
part_d(Q, offset)
part_e(k_q, E, kbest)

(a) MACHINE LEARNING - Support Vector Regression
Train rows: 420  Test rows: 180
RMSE = 0.3649   R^2 = 0.9377

Predicted yield (t/ha) for each plot x crop:
Plot |  Wheat |   Rice |  Maize
  1  |  5.969 |  8.812 |  7.580
  2  |  7.319 |  8.129 |  7.267
  3  |  6.359 |  7.593 |  6.147
  4  |  6.153 |  9.729 |  7.348

(b) QUBO FORMULATION + BRUTE FORCE CHECK
QUBO:  E(x) = - sum Y[p,c] x_pc + A sum_p (sum_c x_pc - 1)^2
              + B sum_(p,q neighbours) sum_c x_pc x_qc
Penalties A = B = 19.458  (= 2 x max predicted yield),  constant offset = 77.831
Q matrix (12 x 12, upper triangular):
 [[-25.43  38.92  38.92  19.46   0.     0.     0.     0.     0.     0.     0.     0.  ]
 [  0.   -28.27  38.92   0.    19.46   0.     0.     0.     0.     0.     0.     0.  ]
 [  0.     0.   -27.04   0.     0.    19.46   0.     0.     0.     0.     0.     0.  ]
 [  0.     0.     0.   -26.78  38.92  38.92  19.46   0.     0.     0.     0.     0.  ]
 [  0.     0.     0.     0.   -27.59  38.92   0.    19.46 

Process SpawnProcess-1:
Traceback (most recent call last):
Process SpawnProcess-2:
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/concurrent/futures/process.py", line 242, in _process_worker
    call_item = call_queue.get(block=True)
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py", line 120, in get
    return _ForkingPickler.loads(res)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^
AttributeError: Can't get attribute 'energy_chunk' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
Traceback (most recent call last):
  File "/Library/Fra

BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.